# Cypher Query Patterns

Common Cypher patterns illustrated against a generic **Project → Company / Person / Risk** knowledge graph.

The setup cell builds an in-memory graph — no external database required.


In [ ]:
# ── Setup: imports, models, schema, ingest ──────────────────────────────────
import sys
import tempfile
from pathlib import Path

from pydantic import BaseModel, Field
from genai_graph.kg.schema import GraphNode, GraphRelation, GraphSchema
from genai_graph.kg.ingest import create_graph, restart_database


def _show_html(html: str, stem: str = "viz") -> None:
    path = Path(tempfile.mkdtemp()) / f"{stem}.html"
    path.write_text(html, encoding="utf-8")
    if "ipykernel" in sys.modules:
        from IPython.display import HTML, display  # noqa: PLC0415
        display(HTML(html))
    else:
        print(f"Saved: {path}")


# --- Domain models ---

class Company(BaseModel):
    name: str
    sector: str | None = None
    country: str = "France"

class Person(BaseModel):
    name: str
    role: str | None = None
    seniority: str = "mid"

class Risk(BaseModel):
    description: str
    impact: str = "medium"
    probability: str = "low"

class Project(BaseModel):
    title: str
    status: str = "active"
    budget_k: int = 0
    client: Company
    team: list[Person] = Field(default_factory=list)
    risks: list[Risk] = Field(default_factory=list)


# --- Schema ---

project_node = GraphNode(node_class=Project, name_from="title", key_from="title")
company_node = GraphNode(node_class=Company, name_from="name",  key_from="name")
person_node  = GraphNode(node_class=Person,  name_from="name",  key_from="name")
risk_node    = GraphNode(node_class=Risk,    name_from="description", key_from="AUTO_ID")

schema = GraphSchema(
    root_model_class=Project,
    nodes=[project_node, company_node, person_node, risk_node],
    relations=[
        GraphRelation(from_node=project_node, to_node=company_node, name="FOR_CLIENT"),
        GraphRelation(from_node=project_node, to_node=person_node,  name="HAS_MEMBER"),
        GraphRelation(from_node=project_node, to_node=risk_node,    name="HAS_RISK"),
    ],
)

# --- Ingest sample data ---

backend = restart_database()

projects = [
    Project(title="Cloud Migration",    status="in-progress", budget_k=500,
            client=Company(name="Acme Corp",    sector="Retail",  country="France"),
            team=[Person(name="Alice",  role="Lead",     seniority="senior"),
                  Person(name="Bob",    role="Engineer", seniority="mid")],
            risks=[Risk(description="Data loss",     impact="high",   probability="low"),
                   Risk(description="Scope creep",   impact="medium", probability="high")]),
    Project(title="ERP Modernisation",  status="planning",    budget_k=200,
            client=Company(name="GlobalSoft",   sector="Finance", country="Germany"),
            team=[Person(name="Alice",  role="Lead",      seniority="senior"),
                  Person(name="Carol",  role="Architect", seniority="senior")],
            risks=[Risk(description="Budget overrun", impact="high", probability="medium")]),
    Project(title="Data Platform",      status="active",      budget_k=350,
            client=Company(name="Acme Corp",    sector="Retail",  country="France"),
            team=[Person(name="Bob",    role="Engineer", seniority="mid"),
                  Person(name="Diana",  role="Analyst",  seniority="junior")],
            risks=[Risk(description="Vendor lock-in", impact="medium", probability="low")]),
    Project(title="Security Audit",     status="active",      budget_k=80,
            client=Company(name="SecureCo",     sector="Finance", country="France"),
            team=[Person(name="Carol",  role="Architect", seniority="senior")],
            risks=[]),
]
for p in projects:
    create_graph(backend, p, schema)

def run(cypher: str, title: str = "") -> None:
    """Execute Cypher and print as a table."""
    df = backend.execute_get_as_df(cypher)
    if title:
        print(f"\n{'─'*60}\n{title}\n{'─'*60}")
    print(df.to_string(index=False) if not df.empty else "(no results)")

print("✅ Graph ready — 4 projects, 4 companies, 4 persons, 4 risks")


## Basic Patterns

In [ ]:
# Count nodes by type
for label in ["Project", "Company", "Person", "Risk"]:
    df = backend.execute_get_as_df(f"MATCH (n:{label}) RETURN count(n) AS cnt")
    print(f"  {label}: {df['cnt'].iloc[0]}")


In [ ]:
# All projects with status
run("MATCH (p:Project) RETURN p.title, p.status, p.budget_k ORDER BY p.title",
    "All projects")


## Traversal Patterns

In [ ]:
# 1-hop: projects → client
run(
    "MATCH (p:Project)-[:FOR_CLIENT]->(c:Company) RETURN p.title, c.name, c.sector",
    "Projects with client",
)


In [ ]:
# 2-hop: persons → projects → companies  ("who works for which client?")
run(
    """
    MATCH (m:Person)<-[:HAS_MEMBER]-(p:Project)-[:FOR_CLIENT]->(c:Company)
    RETURN m.name AS person, p.title AS project, c.name AS client
    ORDER BY m.name, p.title
    """,
    "Person → Project → Company",
)


In [ ]:
# Multi-relation traversal: all neighbours of a project
# (Ladybug uses _label on the relationship object; query explicitly by type)
for rel_name in ["FOR_CLIENT", "HAS_MEMBER", "HAS_RISK"]:
    run(
        f"""
        MATCH (p:Project)-[:{rel_name}]->(n)
        WHERE p.title = 'Cloud Migration'
        RETURN '{rel_name}' AS relation, n.name AS neighbour
        """,
        f"Cloud Migration → {rel_name}",
    )


## Aggregation Patterns

In [ ]:
# Team size per project
run(
    """
    MATCH (p:Project)-[:HAS_MEMBER]->(m:Person)
    RETURN p.title, count(m) AS team_size ORDER BY team_size DESC
    """,
    "Team size per project",
)


In [ ]:
# Persons appearing in more than one project
run(
    """
    MATCH (p:Project)-[:HAS_MEMBER]->(m:Person)
    WITH m.name AS person, collect(p.title) AS projects, count(p) AS cnt
    WHERE cnt > 1
    RETURN person, cnt, projects
    """,
    "Persons in multiple projects",
)


In [ ]:
# Total budget per client
run(
    """
    MATCH (p:Project)-[:FOR_CLIENT]->(c:Company)
    RETURN c.name AS client, sum(p.budget_k) AS total_budget_k
    ORDER BY total_budget_k DESC
    """,
    "Budget by client",
)


## Filtering Patterns

In [ ]:
# Projects with high-impact risks
run(
    """
    MATCH (p:Project)-[:HAS_RISK]->(r:Risk)
    WHERE r.impact = 'high'
    RETURN p.title, r.description, r.probability
    ORDER BY p.title
    """,
    "High-impact risks",
)


In [ ]:
# Active projects with budget > 300k
run(
    """
    MATCH (p:Project) WHERE p.status = 'active' AND p.budget_k > 300
    RETURN p.title, p.status, p.budget_k
    """,
    "Active large projects",
)


In [ ]:
# French clients with finance sector  (STARTS WITH / IN patterns)
run(
    """
    MATCH (c:Company)
    WHERE c.country = 'France' OR c.sector = 'Finance'
    RETURN c.name, c.sector, c.country
    ORDER BY c.name
    """,
    "French or finance companies",
)


## KG Visualisation

In [ ]:
from genai_graph.kg.export.html import generate_html

kg_html = generate_html(connection=backend)
_show_html(kg_html, "cypher_kg")
print(f"KG HTML: {len(kg_html):,} bytes")
